<a href="https://colab.research.google.com/github/asavine/presentations/blob/main/american_average.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

$n$: number of days

$i$: day $i$, $i=0$: today, $i=n$: maturity

$S_i$: fixing on day $i$

$A_i$: running average on day $i$ (excluding day $0$) $A_i = \frac{1}{i} \sum_{j=1}^i S_j$

normal model: $S_{i+1} - S_i \sim N(\mu_d, \sigma_d)$ where drift and vol are expressed in bp per day

---

european option: on day $k$, exercise into $A_k$ or keep into $A_n$

i.e. payoff on day $k$ is $V_k = \max(A_k, E_k[A_n])$ and price today is $V_0 = E_0[V_k]$

$V_k = A_k + max(0, E_k[A_n] - A_k) = A_k + max(0, X_k)$

where $X_k = E_k[A_n] - A_k \sim N(\mu_x, \sigma_x)$

and $\mu_x = E[A_n - A_k] = \frac{n-k}{2}\mu_d$ , $\sigma_x^2 = \left(\frac{n-k}{n}\right) ^ 2 \frac{(k-1)(2k-1)}{6k}\sigma_d^2$

so $V_0 = E[A_k] + B(\mu_x, \sigma_x)$

where $B$ is Bachelier's formula for a zero call on a normal asset with mean $\mu$ and variance $\sigma^2$

and $V_0 - E[A_n] = B(\mu_x, \sigma_x) - E[A_n - A_k] = B(\mu_x, \sigma_x) - \mu_x$

is the premium over plain average

---

american option: exercise on any day so $V_n = A_n$ and $V_i = \max(A_i, E_i[V_{i+1}])$ resolved in code with Gauss-Hermite

In [1]:
import numpy as np
from scipy.stats import norm
days_per_year = 365

# returns V0 - E[An] in bps
# the premium, over n-day average, for the option of fixing the running average on day k
def european_average(
    k,    # option maturity in days
    n=30, # total number of days
    sigma_ann=100.0,  # annual volatility in bp per annum
    mu_ann=0.0,       # drift in bp per annum
):

    # daily volatility and drift
    sigma_d = sigma_ann / np.sqrt(days_per_year)
    mu_d = mu_ann / days_per_year

    # x = E_k[a_n] - a_k
    mu_x = (n - k) / 2 * mu_d
    var_x = (1 - k / n) ** 2 * (k - 1) * (2 * k - 1) / (6 * k) * sigma_d * sigma_d
    sigma_x = np.sqrt(var_x)

    # avoid numerical error with intrinsic
    sigma_x = max(sigma_x, 1e-8)

    # bachelier
    d = mu_x / sigma_x
    bach = sigma_x * norm.pdf(d) + mu_x * norm.cdf(d)

    # premium over a_n
    prem = bach - mu_x

    # round to bps with 2 decimals
    return float(round(prem, 2))


def european_table(
    n=30,
    sigma_ann=100.0,
    mu_ann=-0.0
):
    return {k: european_average(k, n, sigma_ann, mu_ann) for k in range(1, n + 1)}

all_europeans = european_table()
most_expensive = max(all_europeans, key=all_europeans.get)

all_europeans, f"most expensive: {all_europeans[most_expensive]} bps on day {most_expensive}"

({1: 0.0,
  2: 0.97,
  3: 1.4,
  4: 1.69,
  5: 1.91,
  6: 2.06,
  7: 2.18,
  8: 2.26,
  9: 2.32,
  10: 2.35,
  11: 2.36,
  12: 2.35,
  13: 2.32,
  14: 2.28,
  15: 2.22,
  16: 2.14,
  17: 2.06,
  18: 1.96,
  19: 1.85,
  20: 1.73,
  21: 1.6,
  22: 1.46,
  23: 1.31,
  24: 1.14,
  25: 0.97,
  26: 0.8,
  27: 0.61,
  28: 0.41,
  29: 0.21,
  30: 0.0},
 'most expensive: 2.36 bps on day 11')

In [6]:
# note: the grid is over Pi = Vi - Ei[An]

def american_average(
    n=30,
    sigma_ann=100,
    mu_ann=0.0,
    # for sanity check: k > 0 means price the european maturity k, k == 0 means price the american
    k=0,
    # numerical grid parameters
    m_gh=64,
    L=7,
    Ngrid=1001
):
    # daily volatility and drift
    sigma_d = sigma_ann / np.sqrt(days_per_year)
    mu_d = mu_ann / days_per_year

    # grid
    width = L * sigma_d * np.sqrt(n) + np.abs(mu_d) * n
    dgrid = np.linspace(-width, width, Ngrid)

    # terminal: p_n(d) = V_n - E_n[A_n] = 0
    p_next = np.zeros_like(dgrid)

    # Gauss–Hermite for Z~N(0,1)
    xh, wh = np.polynomial.hermite.hermgauss(m_gh)
    z = np.sqrt(2.0) * xh
    eps = mu_d + sigma_d * z
    w = wh / np.sqrt(np.pi)

    # backward induction: t = n-1 down to 0
    for t in range(n - 1, -1, -1):
        if t == 0:
            # D1 = 0 identically (A1 = S1), so p0 = E[p1(D1)] = p1(0)
            p0 = float(np.interp(0.0, dgrid, p_next))
            p_next = np.full_like(dgrid, p0)
            continue

        alpha = t / (t + 1.0)

        # next state: d_{t+1} = alpha * (d_t + eps)
        D_next = alpha * (dgrid[:, None] + eps[None, :])

        # continuation value
        P = np.interp(D_next, dgrid, p_next)
        cont = P @ w

        # exercise premium over E_t[A_n]:
        # ex_t(d) = A_t - E_t[A_n] = - (n-t)/n * ( d + mu_d*(n-t+1)/2 )
        ex = - (n - t) / n * (dgrid + mu_d * (n - t + 1) / 2.0)

        # this is where the sanity check is performed, the rest of the code is unchanged
        p_next = np.maximum(ex, cont) if k == 0 or t == k else cont

    return float(round(np.interp(0.0, dgrid, p_next), 2))

def american_table(
    n=30,
    sigma_ann=100.0,
    mu_ann=-0.0
):
    europeans = np.array([american_average(n, sigma_ann, mu_ann, k) for k in range(1, n + 1)])
    sanity_check = np.array([european_average(k, n, sigma_ann, mu_ann) for k in range(1, n + 1)])
    all = np.vstack((np.arange(1, n+1)[:None], europeans[:None], sanity_check[:None])).T
    maxerr = float(np.round(np.max(np.abs(europeans - sanity_check)), 2))

    most_expensive_idx = int(np.argmax(europeans)) + 1
    most_expensive = float(np.round(europeans[most_expensive_idx - 1], 2))
    american = american_average(n, sigma_ann, mu_ann)
    switch = float(np.round(american - most_expensive))
    return {
        "europeans": all,
        "error": maxerr,
        "most_expensive": (most_expensive_idx, most_expensive),
        "american": american,
        "switch": switch
    }

american_table()


{'europeans': array([[ 1.  ,  0.  ,  0.  ],
        [ 2.  ,  0.98,  0.97],
        [ 3.  ,  1.4 ,  1.4 ],
        [ 4.  ,  1.69,  1.69],
        [ 5.  ,  1.91,  1.91],
        [ 6.  ,  2.07,  2.06],
        [ 7.  ,  2.18,  2.18],
        [ 8.  ,  2.27,  2.26],
        [ 9.  ,  2.32,  2.32],
        [10.  ,  2.35,  2.35],
        [11.  ,  2.36,  2.36],
        [12.  ,  2.35,  2.35],
        [13.  ,  2.32,  2.32],
        [14.  ,  2.28,  2.28],
        [15.  ,  2.22,  2.22],
        [16.  ,  2.15,  2.14],
        [17.  ,  2.06,  2.06],
        [18.  ,  1.96,  1.96],
        [19.  ,  1.85,  1.85],
        [20.  ,  1.73,  1.73],
        [21.  ,  1.6 ,  1.6 ],
        [22.  ,  1.46,  1.46],
        [23.  ,  1.31,  1.31],
        [24.  ,  1.14,  1.14],
        [25.  ,  0.97,  0.97],
        [26.  ,  0.8 ,  0.8 ],
        [27.  ,  0.61,  0.61],
        [28.  ,  0.41,  0.41],
        [29.  ,  0.21,  0.21],
        [30.  ,  0.  ,  0.  ]]),
 'error': 0.01,
 'most_expensive': (11, 2.36),
 'americ